# AFS Validate

Scans extracted financial data for anomalies:
- **YoY swing** — amount changes >10% between consecutive fiscal years
- **Digit mismatch** — digit count differs by ≥2 (strong OCR/LLM misread signal)

Flags are written to `COMMON.REVIEW_QUEUE` for analyst triage.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '../src')

from afs.snowflake_io import cursor_from_session
from afs.validate import validate_org, write_flags_to_review_queue, summarize_flags

In [ ]:
%%sql -r orgs
SELECT ORG_ID, ORG_CODE, LEGAL_NAME
  FROM AUDITED_FINANCIALS.COMMON.ORG_REGISTRY
 ORDER BY LEGAL_NAME

In [ ]:
ORG_CODE = orgs['ORG_CODE'].iloc[0]
org_id = orgs.loc[orgs['ORG_CODE'] == ORG_CODE, 'ORG_ID'].iloc[0]
print(f'Validating: {ORG_CODE} ({org_id})')

In [ ]:
with cursor_from_session(session, commit_on_exit=False) as cur:
    flags = validate_org(cur, org_id)

print(summarize_flags(flags))

In [ ]:
df_flags = pd.DataFrame(flags)
if not df_flags.empty:
    df_flags = df_flags.sort_values(['reason', 'statement', 'native_label'])
    display_cols = ['reason', 'statement', 'native_label', 'fy_label', 'amount', 'detail']
    print(df_flags[display_cols].to_string(index=False))
else:
    print('No flags found.')

In [ ]:
digit_flags = [f for f in flags if f['reason'] == 'digit_mismatch']
if digit_flags:
    print(f'{len(digit_flags)} CRITICAL digit-mismatch flag(s) — likely OCR/LLM misread:')
    for f in digit_flags:
        print(f"  {f['statement']}.{f['native_label']}  {f['detail']}")
else:
    print('No digit-mismatch flags.')

In [ ]:
WRITE_TO_REVIEW_QUEUE = False

if WRITE_TO_REVIEW_QUEUE and flags:
    with cursor_from_session(session) as cur:
        n = write_flags_to_review_queue(cur, org_id, flags)
    print(f'Wrote {n} flag(s) to COMMON.REVIEW_QUEUE.')
else:
    print(f'{len(flags)} flag(s) ready. Set WRITE_TO_REVIEW_QUEUE = True to persist.')